In [5]:
import os 
import json 
import pandas as pd 
from tqdm import tqdm 

In [6]:
#folder containing all json folders
json_folder=r"C:\Users\R Sujit\Downloads\all_male_json"

#getting all json files 
json_files=[f for f in os.listdir(json_folder) if f.endswith(".json")]
print("Total json files found:",len(json_files))

Total json files found: 17824


In [7]:
all_matches=[]
invalid_files=[]
for file_name in tqdm(json_files):
    file_path=os.path.join(json_folder,file_name)
    try:
        with open (file_path,"r",encoding="utf-8")as f:
            match=json.load(f)
            all_matches.append(match)
    except Exception:
        invalid_files.append(file_name)
print("Total files :",len(json_files))
print("Valid matches:",len(all_matches))
print("invalid files:",len(invalid_files))

100%|██████████| 17824/17824 [04:06<00:00, 72.38it/s] 

Total files : 17824
Valid matches: 17824
invalid files: 0


In [20]:
match_data=[]
for file_name,match in tqdm(zip(json_files,all_matches),total=len(all_matches)):
    info=match.get("info",{})
    outcome=info.get("outcome",{})
    toss=info.get("toss",{})
    event=info.get("event",{})
    teams=info.get("teams",[None,None])
    row={"Match_ID":file_name.replace(".json",""),
         "Date":info.get("dates",[None])[0],
         "Num_Days": len(info.get("dates",[])),
         "Match_Type":info.get("match_type"),
         "Gender":info.get("gender"),
         "Team1":teams[0] if len(teams)>0 else None,
         "Team2":teams[1] if len(teams)>1 else None,
         "Venue":info.get("Venue"),
         "City":info.get("city"),
         "Season":info.get("season"),
         "Event_Name":info.get("name"),
         "Balls_per_over":info.get("balls_per_over",6),
         "Overs":info.get("overs"),
         "Toss_Winner":toss.get("winner"),
         "Toss_Decision":toss.get("decision"),
         "Winner":outcome.get("winner"),
         "Result_Type":outcome.get("result"),
         "Win_By_Runs":outcome.get("by",{}).get("runs"),
         "Win_By_Wickets":outcome.get("by",{}).get("wickets"),
         "Player_of_Matche":",".join(info.get("player_of_match",[])),
         "Num_Innings":len(match.get("innings",[])),
         "Umpires":",".join(info.get("officials",{}).get("umpires",[])),
        }
    match_data.append(row)
match_df=pd.DataFrame(match_data)
print("Shape:",match_df.shape)
print("\nMatch type distribution:")
print(match_df["Match_Type"].value_counts())
print("\nMissing values per column:")
print(match_df.isnull().sum())
match_df.head()


100%|██████████| 17824/17824 [00:00<00:00, 51513.22it/s]


Shape: (17824, 22)

Match type distribution:
Match_Type
T20     10394
ODI      2552
MDM      2179
ODM      1573
Test      886
IT20      240
Name: count, dtype: int64

Missing values per column:
Match_ID                0
Date                    0
Num_Days                0
Match_Type              0
Gender                  0
Team1                   0
Team2                   0
Venue               17824
City                 1490
Season                  0
Event_Name          17824
Balls_per_over          0
Overs                3065
Toss_Winner             0
Toss_Decision           0
Winner               1551
Result_Type         16273
Win_By_Runs          9719
Win_By_Wickets       9661
Player_of_Matche        0
Num_Innings             0
Umpires                 0
dtype: int64


,Match_ID,Date,Num_Days,Match_Type,Gender,Team1,Team2,Venue,City,Season,...,Overs,Toss_Winner,Toss_Decision,Winner,Result_Type,Win_By_Runs,Win_By_Wickets,Player_of_Matche,Num_Innings,Umpires
0,1000851,2016-11-03,5,Test,male,Australia,South Africa,None,Perth,2016/17,...,NaN,South Africa,bat,South Africa,None,177.0,NaN,K Rabada,4,"Aleem Dar,NJ Llong"
1,1000853,2016-11-12,4,Test,male,Australia,South Africa,None,Hobart,2016/17,...,NaN,South Africa,field,South Africa,None,80.0,NaN,KJ Abbott,3,"Aleem Dar,RA Kettleborough"
2,1000855,2016-11-24,4,Test,male,Australia,South Africa,None,None,2016/17,...,NaN,South Africa,bat,Australia,None,NaN,7.0,UT Khawaja,4,"RA Kettleborough,NJ Llong"
3,1000881,2016-12-15,5,Test,male,Australia,Pakistan,None,Brisbane,2016/17,...,NaN,Australia,bat,Australia,None,39.0,NaN,Asad Shafiq,4,"IJ Gould,RK Illingworth"
4,1000883,2016-12-26,5,Test,male,Australia,Pakistan,None,None,2016/17,...,NaN,Pakistan,bat,Australia,None,18.0,NaN,SPD Smith,3,"IJ Gould,S Ravi"


In [21]:
print(match_df.shape)

(17824, 22)


In [22]:
print("Length of all_matches",len(all_matches))
print("Length of json_files:",len(json_files))
print("Length of match_data:",len(match_data))

Length of all_matches 17824
Length of json_files: 17824
Length of match_data: 17824


In [97]:
batting_data=[]
for match_index, match in enumerate(all_matches):
    info=match["info"]
    match_id=match_index+1
    date=info.get("dates",[None])[0]
    venue=info.get("venue",None)
    city=info.get("city",None)
    match_type=info.get("match_type",None)
    if "innings" not in match:
        continue
    for innings in match["innings"]:
        if "overs" not in innings:
            continue
        batting_team=innings["team"]
        batter_stats={}
        for over in innings["overs"]:
            for delivery in over["deliveries"]:
                batter=delivery["batter"]
                if batter is None:
                    continue
                if batter not in batter_stats:
                    batter_stats[batter]={
                        "Runs":0,
                        "Balls":0,
                        "Fours":0,
                        "Sixes":0,
                        "Out":0
                    }
                runs=delivery["runs"]["batter"]
                batter_stats[batter]["Runs"]+=runs
                batter_stats[batter]["Balls"]+=1
                if runs==4:
                    batter_stats[batter]["Fours"]+=1
                elif runs==6:
                    batter_stats[batter]["Sixes"]+=1
                if "wickets" in delivery:
                    for wicket in delivery["wickets"]:
                        if wicket["player_out"]==batter:
                            batter_stats[batter]["Out"]=1
        for batter,stats in batter_stats.items():
            strike_rate=0
            if stats["Balls"]>0:
                strike_rate=round((stats["Runs"]/stats["Balls"])*100,2)
                batting_data.append({
                    "Match ID":match_id,
                    "Date":date,
                    "Venue":venue,
                    "City":city,
                    "Match_Type":match_type,
                    "Batting_Team":batting_team,
                    "Batter":batter,
                    "Runs":stats["Runs"],
                    "Balls":stats["Balls"],
                    "Fours":stats["Fours"],
                    "Sixes":stats["Sixes"],
                    "Out":stats["Out"],
                    "Strike_Rate":strike_rate
                                           })
           
                        

In [99]:
batting_df=pd.DataFrame(batting_data)
batting_df.head()
print(batting_df.shape)
output_path=r"C:\Users\R Sujit\Downloads\batting_dataset.csv"
batting_df.to_csv(output_path,index=False)
print("Batting dataset savced successfully!")
print(batting_df["Match ID"].isnull().sum())

(348062, 13)
Batting dataset savced successfully!
0


In [46]:
print("Total Matches:",len(all_matches))
print("Batting Records:", len(batting_data))
print(batting_df.head())

Total Matches: 17824
Batting Records: 348062
   Match ID        Date                                         Venue   City  \
0    2230.0  2016-11-03  Western Australia Cricket Association Ground  Perth   
1    2230.0  2016-11-03  Western Australia Cricket Association Ground  Perth   
2    2230.0  2016-11-03  Western Australia Cricket Association Ground  Perth   
3    2230.0  2016-11-03  Western Australia Cricket Association Ground  Perth   
4    2230.0  2016-11-03  Western Australia Cricket Association Ground  Perth   

  Match_Type  Batting_Team        Batter  Runs  Balls  Fours  Sixes  Out  \
0       Test  South Africa       SC Cook     0      4      0      0    1   
1       Test  South Africa       HM Amla     0      5      0      0    1   
2       Test  South Africa       D Elgar    12     22      2      0    1   
3       Test  South Africa     JP Duminy    11     23      0      0    1   
4       Test  South Africa  F du Plessis    37     74      5      0    1   

   Strike_Rate  


In [125]:
bowling_data=[]
for match_index, match in enumerate(all_matches):
    match_id=match_index +1
    info=match.get("info",{})
    date=info.get("dates",["Unknow"])[0]
    venue=info.get("venue","Unknown")
    city=info.get("city","Unknown")
    match_type=info.get("match_type","Unknown")
    innings_list=match.get("innings",[])
    bowler_stats={}
    for innings in innings_list:
        if "overs" not in innings:
            continue
        bowling_team=innings.get("team","Unknown")
        for over in innings["overs"]:
            for delivery in over["deliveries"]:
                bowler=delivery.get("bowler")
                if bowler is None:
                    continue
                runs=delivery["runs"]["total"]
                if bowler not in bowler_stats:
                    bowler_stats[bowler]={
                        "balls":0,
                        "runs":0,
                        "wickets":0,
                    }
                bowler_stats[bowler]["balls"]+=1
                bowler_stats[bowler]["runs"]+=runs
                if "wickets" in delivery:
                    bowler_stats[bowler]["wickets"]+=len(delivery["wickets"])
    for bowler, stats in bowler_stats.items():
        overs=stats["balls"]/6
        economy=round(stats["runs"]/overs,2)if overs>0 else 0
        bowling_data.append({
            "Match_ID":match_id,
            "Date":date,
            "Venue":venue,
            "City":city,
            "Match_Type":match_type,
            "Bowling_Team":bowling_team,
            "Bowler":bowler,
            "Balls":stats["balls"],
            "Overs":overs,
            "Runs_Conceded":stats["runs"],
            "Wickets":stats["wickets"],
            "Economy":economy
        })
                        
bowling_df=pd.DataFrame(bowling_data)
print(bowling_df.shape)
bowling_df.head()
               


(213734, 12)


,Match_ID,Date,Venue,City,Match_Type,Bowling_Team,Bowler,Balls,Overs,Runs_Conceded,Wickets,Economy
0,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,Australia,MA Starc,301,50.166667,192,5,3.83
1,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,Australia,JR Hazlewood,325,54.166667,186,5,3.43
2,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,Australia,PM Siddle,228,38.000000,98,3,2.58
3,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,Australia,MR Marsh,196,32.666667,105,2,3.21
4,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,Australia,NM Lyon,264,44.000000,189,2,4.30


In [126]:
output_path=r"C:\Users\R Sujit\Downloads\bowling_dataset.csv"
bowling_df.to_csv(output_path,index=False)
print("Saved successfully!")
print(output_path)

Saved successfully!
C:\Users\R Sujit\Downloads\bowling_dataset.csv


In [81]:
fielding_data=[]
for match_index,match in enumerate(all_matches):
    info=match.get("info",{})
    match_id=match_index+1
    date=info.get("dates",["Unknown"])[0]
    venue=info.get("venue","Unknown")
    city=info.get("city","Unknown")
    match_type=info.get("match_type","Unknown")
    innings_list=match.get("innings",[])
    for innings in innings_list:
        if "overs" not in innings:
            continue
        fielding_team=innings.get("team","Unknown")
        for over in innings["overs"]:
            for delivery in over["deliveries"]:
                if "wickets" not in delivery:
                    continue
                for wicket in delivery["wickets"]:
                    dismissal=wicket.get("kind","Unknown")
                    batter=wicket.get("player_out","Unknown")
                    if "fielders" in wicket:
                        for fielder in wicket["fielders"]:
                            fielding_data.append({
                                "Match_ID":match_id,
                                "Date":date,
                                "Venue":venue,
                                "City":city,
                                "Match_Type":match_type,
                                "Fielding_Team":fielding_team,
                                "Fielder":fielder.get("name","Unknown"),
                                "Dismissal_Type":dismissal,
                                "Batter_Out":batter
                            })
fielding_df=pd.DataFrame(fielding_data)
print(fielding_df.shape)
fielding_df.head(20)

(195068, 9)


,Match_ID,Date,Venue,City,Match_Type,Fielding_Team,Fielder,Dismissal_Type,Batter_Out
0,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,South Africa,MR Marsh,caught,SC Cook
1,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,South Africa,SPD Smith,caught,HM Amla
2,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,South Africa,PM Nevill,caught,D Elgar
3,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,South Africa,PM Nevill,caught,JP Duminy
4,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,South Africa,AC Voges,caught,F du Plessis
5,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,South Africa,SE Marsh,caught,T Bavuma
6,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,South Africa,DA Warner,caught,KA Maharaj
7,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,South Africa,SE Marsh,caught,Q de Kock
8,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,Australia,HM Amla,caught,DA Warner
9,1,2016-11-03,Western Australia Cricket Association Ground,Perth,Test,Australia,F du Plessis,caught,MA Starc


In [82]:
output_path=r"C:\Users\R Sujit\Downloads\fielding_dataset.csv"
fielding_df.to_csv(output_path,index=False)
print("Fielding dataset saved successfully!")

Fielding dataset saved successfully!


In [83]:
print(batting_df.columns.tolist())

['Match ID', 'Date', 'Venue', 'City', 'Match_Type', 'Batting_Team', 'Batter', 'Runs', 'Balls', 'Fours', 'Sixes', 'Out', 'Strike_Rate']


In [85]:
runs_summary=batting_df.groupby("Batter")["Runs"].sum()
print(runs_summary.head(10))

Batter
A Adey             13
A Ahmadhel          8
A Alexander        50
A Amado            19
A Amburose          0
A Anandakumara      9
A Andrews           4
A Anlezark         10
A Aravinddaraj      0
A Ashish Reddy    415
Name: Runs, dtype: int64


In [88]:
balls_summary=batting_df.groupby("Batter")["Balls"].sum()
print(balls_summary.head(10))

Batter
A Adey             17
A Ahmadhel         27
A Alexander        35
A Amado            28
A Amburose          6
A Anandakumara     22
A Andrews           6
A Anlezark         24
A Aravinddaraj      3
A Ashish Reddy    326
Name: Balls, dtype: int64


In [89]:
player_summary=pd.DataFrame()
player_summary["Runs"]=batting_df.groupby("Batter")["Runs"].sum()
player_summary["Balls"]=batting_df.groupby("Batter")["Balls"].sum()
player_summary.head(10)

,Runs,Balls
Batter,,
A Adey,13,17
A Ahmadhel,8,27
A Alexander,50,35
A Amado,19,28
A Amburose,0,6
A Anandakumara,9,22
A Andrews,4,6
A Anlezark,10,24
A Aravinddaraj,0,3


In [91]:
player_summary ["Matches"]=batting_df.groupby("Batter")["Match ID"].nunique()
player_summary.head(10)

,Runs,Balls,Matches
Batter,,,
A Adey,13,17,0
A Ahmadhel,8,27,4
A Alexander,50,35,8
A Amado,19,28,2
A Amburose,0,6,2
A Anandakumara,9,22,0
A Andrews,4,6,1
A Anlezark,10,24,0
A Aravinddaraj,0,3,0


In [100]:
import numpy as np 

#batting summary
player_summary=batting_df.groupby("Batter").agg(
    Matches=("Match ID","nunique"),
    Innings=("Match ID","count"),
    Runs=("Runs","sum"),
    Balls=("Balls","sum"),
    Fours=("Fours","sum"),
    Sixes=("Sixes","sum"),
    Outs=("Out","sum"),
).reset_index()
#not outs
player_summary["Not_Outs"]=(
    player_summary["Innings"]-player_summary["Outs"]
)
#batting average 
player_summary["Batting_Average"]=np.where(
    player_summary["Outs"]>0,
    player_summary["Runs"]/player_summary["Outs"],
    player_summary["Runs"]
)
#strike rate 
player_summary["Strike_Rate"]=np.where(
    player_summary["Balls"]>0,
    (player_summary["Runs"]/player_summary["Balls"])*100,
     0
)
#boundary percentage
player_summary["Boundary_Percentage"]=np.where(
    player_summary["Balls"]>0,
    ((player_summary["Fours"]+player_summary["Sixes"])/player_summary["Balls"])*100,
    0
)
#round values
player_summary["Batting_Average"]=player_summary["Batting_Average"].round(2)
player_summary["Strike_Rate"]=player_summary["Strike_Rate"].round(2)
player_summary["Boundary_Percentage"]=player_summary["Boundary_Percentage"].round(2)
print("Player Summary Shape:",player_summary.shape)
player_summary.head(10)
     

Player Summary Shape: (9283, 12)


,Batter,Matches,Innings,Runs,Balls,Fours,Sixes,Outs,Not_Outs,Batting_Average,Strike_Rate,Boundary_Percentage
0,A Adey,1,1,13,17,2,0,1,0,13.00,76.47,11.76
1,A Ahmadhel,4,4,8,27,1,0,4,0,2.00,29.63,3.70
2,A Alexander,8,8,50,35,2,4,5,3,10.00,142.86,17.14
3,A Amado,2,2,19,28,2,0,2,0,9.50,67.86,7.14
4,A Amburose,2,2,0,6,0,0,2,0,0.00,0.00,0.00
5,A Anandakumara,2,2,9,22,0,0,2,0,4.50,40.91,0.00
6,A Andrews,1,1,4,6,1,0,1,0,4.00,66.67,16.67
7,A Anlezark,4,4,10,24,0,0,1,3,10.00,41.67,0.00
8,A Aravinddaraj,1,1,0,3,0,0,0,1,0.00,0.00,0.00
9,A Ashish Reddy,35,35,415,326,24,18,26,9,15.96,127.30,12.88


In [101]:
batting_df[["Match ID","Batter"]].head(20)
print(batting_df["Match ID"].unique()[:10])

[ 1  2  3  4  5  6  7  8  9 10]


In [102]:
#checking
print(batting_df["Match ID"].isnull().sum())
batting_df[batting_df["Match ID"].isnull()].head(20)
print(batting_df["Match ID"].dtype)

0
int64


In [103]:
batting_df[batting_df["Match ID"].isnull()].head(20)

,Match ID,Date,Venue,City,Match_Type,Batting_Team,Batter,Runs,Balls,Fours,Sixes,Out,Strike_Rate


In [116]:
print(type(bowling_df))
print(bowling_df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
['Match_ID', 'Date', 'Venue', 'City', 'Match_Type', 'Bowling_Team', 'Bowler', 'Balls', 'Overs', 'Runs_Conceded', 'Wickets', 'Economy']


In [117]:
print(bowling_df.head())

   Match_ID        Date                                         Venue   City  \
0         1  2016-11-03  Western Australia Cricket Association Ground  Perth   
1         1  2016-11-03  Western Australia Cricket Association Ground  Perth   
2         1  2016-11-03  Western Australia Cricket Association Ground  Perth   
3         1  2016-11-03  Western Australia Cricket Association Ground  Perth   
4         1  2016-11-03  Western Australia Cricket Association Ground  Perth   

  Match_Type  Bowling_Team        Bowler  Balls  Overs  Runs_Conceded  \
0       Test  South Africa      MA Starc    114   19.0             71   
1       Test  South Africa  JR Hazlewood    102   17.0             74   
2       Test  South Africa     PM Siddle     72   12.0             36   
3       Test  South Africa      MR Marsh     36    6.0             23   
4       Test  South Africa       NM Lyon     60   10.0             38   

   Wickets  Economy  
0        4     3.74  
1        3     4.35  
2        1    

In [110]:
output_path=r"C:\Users\R Sujit\Downloads\batting_summary.csv"
player_summary.to_csv(output_path,index=False)
print("Batting Summary saved successfully!")

Batting Summary saved successfully!


In [135]:
bowling_summary=bowling_df.groupby("Bowler").agg(
    Bowling_Matches=("Match_ID","nunique"),
    Balls_Bowled=("Balls","sum"),
    Runs_Conceded=("Runs_Conceded","sum"),
    Wickets=("Wickets","sum")
).reset_index()

#bowling average
bowling_summary["Bowling_Average"]=np.where(
    bowling_summary["Wickets"]>0,
    bowling_summary["Runs_Conceded"]/bowling_summary["Wickets"],
    np.nan
)

#economy rate 
bowling_summary["Economy"]=np.where(
    bowling_summary["Balls_Bowled"]>0,
    bowling_summary["Runs_Conceded"]/(bowling_summary["Balls_Bowled"]/6),
    np.nan
)

#round value
bowling_summary["Bowling_Average"]=bowling_summary["Bowling_Average"].round(2)
bowling_summary["Economy"]=bowling_summary["Economy"].round(2)
print("Bowling Summary Shape:",bowling_summary.shape)
bowling_summary.head(10)


Bowling Summary Shape: (7192, 7)


,Bowler,Bowling_Matches,Balls_Bowled,Runs_Conceded,Wickets,Bowling_Average,Economy
0,A Adey,1,18,34,1,34.00,11.33
1,A Ahmadhel,2,12,21,1,21.00,10.50
2,A Alexander,8,108,105,9,11.67,5.83
3,A Amado,2,18,20,1,20.00,6.67
4,A Andrews,3,62,64,5,12.80,6.19
5,A Anlezark,4,203,181,5,36.20,5.35
6,A Aravinddaraj,9,165,212,6,35.33,7.71
7,A Ashish Reddy,27,399,593,24,24.71,8.92
8,A Ashok,57,5050,3500,107,32.71,4.16
9,A Ashokan,19,387,463,19,24.37,7.18


In [136]:
bowling_summary.to_csv(
    r"C:\Users\R Sujit\Downloads\bowling_summary.csv",
    index=False
)
print("Bowling Summary saved successfully!")

Bowling Summary saved successfully!


In [137]:
fielding_summary=fielding_df.groupby("Fielder").agg(
    Fielding_Matches=("Match_ID","nunique")
).reset_index()

#dismissal types
catches=(
    fielding_df[fielding_df["Dismissal_Type"]=="caught"]
    .groupby("Fielder")
    .size()
)
run_outs=(
    fielding_df[fielding_df["Dismissal_Type"]=="run out"]
    .groupby("Fielder")
    .size()
)
stumpings=(
    fielding_df[fielding_df["Dismissal_Type"]=="stumped"]
    .groupby("Fielder")
    .size()
)
fielding_summary["Catches"]=fielding_summary["Fielder"].map(catches).fillna(0).astype(int)
fielding_summary["Run_Ounts"]=fielding_summary["Fielder"].map(run_outs).fillna(0).astype(int)
fielding_summary["Stumpings"]=fielding_summary["Fielder"].map(stumpings).fillna(0).astype(int)
print("Fielding Summary Shape:",fielding_summary.shape)
fielding_summary.head(10)

Fielding Summary Shape: (7888, 5)


,Fielder,Fielding_Matches,Catches,Run_Ounts,Stumpings
0,A Adey,1,1,0,0
1,A Ahmadhel,1,1,0,0
2,A Alexander,10,9,2,0
3,A Amado,2,2,0,0
4,A Anandakumara,2,2,0,0
5,A Andrews,3,4,0,0
6,A Aravinddaraj,1,1,0,0
7,A Ashish Reddy,11,11,3,0
8,A Ashok,14,13,2,0
9,A Ashokan,9,10,3,0


In [138]:
output_path=r"C:\Users\R Sujit\Downloads\fielding_summary.csv"
fielding_summary.to_csv(output_path,index=False)
print("Fielding Summary saved successfully!")
print("File saved at:",output_path)

Fielding Summary saved successfully!
File saved at: C:\Users\R Sujit\Downloads\fielding_summary.csv


In [143]:
batting_summary=player_summary.rename(columns={"Batter":"Player"})
bowling_summary=bowling_summary.rename(columns={"Bowler":"Player"})
fielding_summary=fielding_summary.rename(columns={"Fielder":"Player"})
player_stats=pd.merge(
    batting_summary,
    bowling_summary,
    on="Player",
    how="outer"
)
player_stats=pd.merge(
    player_stats,
    fielding_summary,
    on="Player",
    how="outer"
)
player_stats=player_stats.fillna(0)
print(player_stats.shape)
player_stats.head(10)

(9836, 22)


,Player,Matches,Innings,Runs,Balls,Fours,Sixes,Outs,Not_Outs,Batting_Average,...,Bowling_Matches,Balls_Bowled,Runs_Conceded,Wickets,Bowling_Average,Economy,Fielding_Matches,Catches,Run_Ounts,Stumpings
0,A Adey,1.0,1.0,13.0,17.0,2.0,0.0,1.0,0.0,13.00,...,1.0,18.0,34.0,1.0,34.00,11.33,1.0,1.0,0.0,0.0
1,A Ahmadhel,4.0,4.0,8.0,27.0,1.0,0.0,4.0,0.0,2.00,...,2.0,12.0,21.0,1.0,21.00,10.50,1.0,1.0,0.0,0.0
2,A Alexander,8.0,8.0,50.0,35.0,2.0,4.0,5.0,3.0,10.00,...,8.0,108.0,105.0,9.0,11.67,5.83,10.0,9.0,2.0,0.0
3,A Amado,2.0,2.0,19.0,28.0,2.0,0.0,2.0,0.0,9.50,...,2.0,18.0,20.0,1.0,20.00,6.67,2.0,2.0,0.0,0.0
4,A Amburose,2.0,2.0,0.0,6.0,0.0,0.0,2.0,0.0,0.00,...,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.0,0.0,0.0
5,A Anandakumara,2.0,2.0,9.0,22.0,0.0,0.0,2.0,0.0,4.50,...,0.0,0.0,0.0,0.0,0.00,0.00,2.0,2.0,0.0,0.0
6,A Andrews,1.0,1.0,4.0,6.0,1.0,0.0,1.0,0.0,4.00,...,3.0,62.0,64.0,5.0,12.80,6.19,3.0,4.0,0.0,0.0
7,A Anlezark,4.0,4.0,10.0,24.0,0.0,0.0,1.0,3.0,10.00,...,4.0,203.0,181.0,5.0,36.20,5.35,0.0,0.0,0.0,0.0
8,A Aravinddaraj,1.0,1.0,0.0,3.0,0.0,0.0,0.0,1.0,0.00,...,9.0,165.0,212.0,6.0,35.33,7.71,1.0,1.0,0.0,0.0
9,A Ashish Reddy,35.0,35.0,415.0,326.0,24.0,18.0,26.0,9.0,15.96,...,27.0,399.0,593.0,24.0,24.71,8.92,11.0,11.0,3.0,0.0


In [144]:
output_path=r"C:\Users\R Sujit\Downloads\Player_Statistics.csv"
player_stats.to_csv(output_path,index=False)
print("Player Statistics Dataset saved successfully!")
print("File saved at:",output_path)

Player Statistics Dataset saved successfully!
File saved at: C:\Users\R Sujit\Downloads\Player_Statistics.csv


In [145]:
print(match_df.columns)

Index(['Match_ID', 'Date', 'Num_Days', 'Match_Type', 'Gender', 'Team1',
       'Team2', 'Venue', 'City', 'Season', 'Event_Name', 'Balls_per_over',
       'Overs', 'Toss_Winner', 'Toss_Decision', 'Winner', 'Result_Type',
       'Win_By_Runs', 'Win_By_Wickets', 'Player_of_Matche', 'Num_Innings',
       'Umpires'],
      dtype='object')


In [149]:
#venue dataset

venue_df=match_df[[
    "Match_ID",
    "Venue",
    "City",
    "Match_Type",
    "Team1",
    "Team2"
]].copy()
np.random.seed(42)
pitch_types=[
    "Batting",
    "Bowling",
    "Balanced",
    "Spin-Friendly",
    "Pace-Friendly"
]
venue_df["Pitch_Type"]=np.random.choice(
    pitch_types,
    size=len(venue_df),
    p=[0.25,0.15,0.35,0.15,0.10]
)
def first_innings_score(match_type):
    if match_type=="T20":
        return np.random.randint(140,221)
    elif match_type=="ODI":
        return np.random.randint(220,381)
    elif match_type=="Test":
        return np.random.randint(250,501)
    else:
        return np.random.randint(180,301)
venue_df["Avg_First_Innings_Score"]=venue_df["Match_Type"].apply(first_innings_score)
print("Venue Dataset Shape:",venue_df.shape)
venue_df.head(10)

Venue Dataset Shape: (17824, 8)


,Match_ID,Venue,City,Match_Type,Team1,Team2,Pitch_Type,Avg_First_Innings_Score
0,1000851,None,Perth,Test,Australia,South Africa,Bowling,476
1,1000853,None,Hobart,Test,Australia,South Africa,Pace-Friendly,362
2,1000855,None,None,Test,Australia,South Africa,Balanced,346
3,1000881,None,Brisbane,Test,Australia,Pakistan,Balanced,300
4,1000883,None,None,Test,Australia,Pakistan,Batting,367
5,1000885,None,None,Test,Australia,Pakistan,Batting,285
6,1000887,None,Brisbane,ODI,Australia,Pakistan,Batting,300
7,1000889,None,None,ODI,Australia,Pakistan,Spin-Friendly,322
8,1000891,None,Perth,ODI,Australia,Pakistan,Balanced,253
9,1000893,None,None,ODI,Australia,Pakistan,Balanced,359


In [150]:
output_path=r"C:\Users\R Sujit\Downloads\venue_dataset.csv"
venue_df.to_csv(output_path,index=False)
print("Venue Dataset saved successfully!")
print("File saved at:",output_path)

Venue Dataset saved successfully!
File saved at: C:\Users\R Sujit\Downloads\venue_dataset.csv


In [151]:
cities=sorted(match_df["City"].dropna().unique())
print("Total Cities:",len(cities))
for city in cities:
    print(city)

Total Cities: 323
Aberdeen
Abu Dhabi
Abuja
Accra
Adelaide
Ahmedabad
Al Amarat
Albergaria
Albury
Alexandra
Alice Springs
Almeria
Amstelveen
Antigua
Apia
Arundel
Auckland
Ayr
Bali
Bandar Kinrara
Bangalore
Bangi
Bangkok
Barbados
Basseterre
Beckenham
Belfast
Belgrade
Bendigo
Bengaluru
Benoni
Bermuda
Birmingham
Bishop's Stortford
Blackpool
Blantyre
Bloemfontein
Bogra
Bready
Bridgetown
Brighton
Brisbane
Bristol
Brondby
Buenos Aires
Bulawayo
Cairns
Canberra
Canterbury
Cape Town
Cardiff
Carrara
Castel
Centurion
Chandigarh
Chattogram
Chelmsford
Cheltenham
Chennai
Chester-le-Street
Chesterfield
Chiang Mai
Chittagong
Christchurch
Coffs Harbour
Coggeshall
Colchester
Colombo
Colwyn Bay
Comber
Coolidge
Copenhagen
Cork
Cuttack
Dallas
Dambulla
Dar-es-Salaam
Darwin
Dasmarinas
Dehra Dun
Delhi
Derby
Derry
Deventer
Dhaka
Dharamsala
Dharmasala
Doha
Dominica
Dreux
Dubai
Dublin
Dundee
Dunedin
Durban
East London
Eastbourne
Edinburgh
Eglinton
Entebbe
Episkopi
Faisalabad
Faridabad
Fatullah
Frinton-on-Sea
Gaboro

In [152]:
#teams appearing in the dataset
teams=sorted(set(match_df["Team1"]).union(set(match_df["Team2"])))
print("Total Teams:",len(teams))
for team in teams :
    print(team)

Total Teams: 356
Abu Dhabi Knight Riders
Ace Capital Cricket Club
Adelaide Strikers
Africa XI
Andhra
Antigua Hawksbills
Antigua and Barbuda Falcons
Argentina
Arunachal Pradesh
Asia XI
Assam
Auckland
Auckland Aces
Australia
Austria
B-Love Kandy
Badureliya Sports Club
Bahamas
Bahrain
Bangladesh
Barbados Royals
Barbados Tridents
Barisal Bulls
Barisal Burners
Baroda
Belgium
Belize
Bengal
Bermuda
Bhutan
Bihar
Biratnagar Kings
Birmingham Bears
Birmingham Phoenix
Bloomfield Cricket and Athletic Club
Boland
Botswana
Brazil
Brisbane Heat
Bulgaria
Burgher Recreation Club
Cambodia
Cameroon
Canada
Canterbury
Cape Cobras
Cape Town Blitz
Cayman Islands
Central Districts
Central Stags
Chandigarh
Chattogram Challengers
Chattogram Royals
Chennai Super Kings
Chhattisgarh
Chilaw Marians Cricket Club
Chile
China
Chittagong Kings
Chittagong Vikings
Chitwan Rhinos
Colombo Cricket Club
Colombo Kings
Colombo Stars
Colombo Strikers
Colts Cricket Club
Comilla Victorians
Cook Islands
Costa Rica
Croatia
Cumilla W

In [153]:
#location dataset
location_df=match_df[["Match_ID","City"]].copy()
unique_locations=(
    location_df[["City"]]
    .drop_duplicates()
    .sort_values("City")
    .reset_index(drop=True)
)
print("Unique Cities:",len(unique_locations))
unique_locations.head(20)

Unique Cities: 324


,City
0,Aberdeen
1,Abu Dhabi
2,Abuja
3,Accra
4,Adelaide
5,Ahmedabad
6,Al Amarat
7,Albergaria
8,Albury
9,Alexandra


In [154]:
output_path=r"C:\Users\R Sujit\Downloads\unique_cities.csv"
unique_locations.to_csv(output_path,index=False)
print("Unique Cities file saved!")

Unique Cities file saved!


In [204]:
#dictionary to map cities to countries

country_map1={
    #Australia
    "Adelaide":"Australia",
    "Brisbane":"Australia",
    "Canberra":"Australia",
    "Hobart":"Australia",
    "Melbourne":"Australia",
    "Perth":"Australia",
    "Sydney":"Australia",
    "Geelong":"Australia",
    "Townsville":"Australia",
    "Darwin":"Australia",
    "Cairns":"Australia",
    "Launceston":"Australia",
    "Gold Coast":"Australia",

    #India
    "Ahmedabad":"India",
    "Bengaluru":"India",
    "Bangalore":"India",
    "Chennai":"India",
    "Delhi":"India",
    "Dharmasala":"India",
    "Hyderabad":"India",
    "Indore":"India",
    "Jaipur":"India",
    "Kolkata":"India",
    "Lucknow":"India",
    "Mohali":"India",
    "Mumbai":"India",
    "Nagpur":"India",
    "Pune":"India",
    "Rajkot":"India",
    "Visakhapatnam":"India",
    "Cuttack":"India",
    "Kanpur":"India",
    "Ranchi":"India",
    "Guwahati":"India",
    "Tiruvananthapuram":"India",

    #England
    "London":"England",
    "Manchester":"England",
    "Leeds":"England",
    "Birmingham":"England",
    "Southampton":"England",
    "Nottingham":"England",
    "Bristol":"England",
    "Cardiff":"England",
    "Chester-le-Street":"England",
    "The Oval":"England",

    #Ireland
    "Dublin":"Ireland",
    "Belfast":"Ireland",
    "Cork":"Ireland",

    #New Zealand
    "Auckland":"New Zealand",
    "Wellington":"New Zealand",
    "Hamilton":"New Zealand",
    "Christchurch":"New Zealand",
    "Dunedin":"New Zealand",
    "Napier":"New Zealand",
    "Tauranga":"New Zealand",
    "Queenstown":"New Zealand",

    #Pakistan
    "Karachi":"Pakistan",
    "Lahore":"Pakistan",
    "Rawalpindi":"Pakistan",
    "Multan":"Pakistan",
    "Faisalabad":"Pakistan",

    #Sri Lanka
    "Colombo":"Sri Lanka",
    "Galle":"Sri Lanka",
    "Kandy":"Sri Lanka",
    "Dambulla":"Sri Lanka",
    "Hambantota":"Sri Lanka",
    "Pallekele":"Sri Lanka",

    #Bangladesh
    "Dhaka":"Bangladesh",
    "Chattogram":"Bangladesh",
    "Chittagong":"Bangladesh",
    "Sylhet":"Bangladesh",
    "Khulna":"Bangladesh",
    "Rajshani":"Bangladesh",
    "Barishal":"Bangladesh",
    "Fatullah":"Bangladesh",

    #Botswana
    "Gaborone":"Botswana",

    #Bhutan
    "Gelephu":"Bhutan",

    #Cayman Islands
    "George Town":"Cayman Islands",
    

    #South Africa
    "Cape Town":"South Africa",
    "Johannesburg":"South Africa",
    "Durban":"South Africa",
    "Centurion":"South Africa",
    "Gqeberha":"South Africa",
    "Port Elizabeth":"South Africa",
    "Bloemfontein":"South Africa",
    "Benoni":"South Africa",

    #Zimbabwe
    "Harare":"Zimbabwe",
    "Bulawayo":"Zimbabwe",

    #UAE
    "Dubai":"United Arab Emirates",
    "Abu Dhabi":"United Arab Emirates",
    "Sharjah":"United Arab Emirates",

    #West Indies
    "Bridgetown":"West Indies",
    "Kingston":"Jamaica",
    "Georgetown":"Guyana",
    "St John's":"Antigua and Barbuda",
    "Basseterre":"Saint Kitts and Nevis",
    "Providence":"Guyana",
    "Tarouba":"Trinidad and Tobago",

    #Nepal
    "Kathmandu":"Nepal",

    #Netherlands
    "Amstelveen":"Netherlands",
    "Deventer":"Netherlands",
    

    #Namibia
    "Windhoek":"Namibia",

    #Scotland
    "Edinburh":"Scotland",
    "Glasgow":"Scotland",

    #USA
    "Dallas":"United States",
    "Lauderhill":"United States",
    "Morrisville":"United States",
    "New York":"United States",
    "Grand Prairie":"United States",
    "Houston":"United States",
}
location_df=unique_locations.copy()
location_df["Country"]=location_df["City"].map(country_map)
location_df.head()

,City,Country
0,Aberdeen,NaN
1,Abu Dhabi,United Arab Emirates
2,Abuja,NaN
3,Accra,NaN
4,Adelaide,Australia


In [205]:
unmatched=location_df[location_df["Country"].isna()]
print("Total unmatched cities:",len(unmatched))
unmatched.head(50)

Total unmatched cities: 230


,City,Country
0,Aberdeen,NaN
2,Abuja,NaN
3,Accra,NaN
6,Al Amarat,NaN
7,Albergaria,NaN
8,Albury,NaN
9,Alexandra,NaN
10,Alice Springs,NaN
11,Almeria,NaN
13,Antigua,NaN


In [206]:
country_map2={
    #Scotland
    "Aberdeen":"Scotland",
    "Aye":"Scotland",
    "Ayr":"Scotland",
    "Dundee":"Scotland",
    "Edinburgh":"Scotland",

    #Nigeria
    "Abuja":"Nigeria",

    #Ghana
    "Accra":"Ghana",

    #Oman
    "Al Amarat":"Oman",

    #Portugal
    "Albergaria":"Portugal",

    #Australia
    "Albury":"Australia",
    "Alice Springs":"Australia",
    "Bendigo":"Australia",
    "Carrara":"Australia",
    "Coffs Harbour":"Australia",

    #New Zealand
    "Alexandra":"New Zealand",
    "Canterbury":"New Zealand",
    "Gisborne":"New Zealand",
    "Invercargill":"New Zealand",
    

    #Spain
    "Almeria":"Spain",

    #Antigua and Barbuda
    "Antigua":"Antigua and Barbuda",
    "Coolidge":"Antigua and Barbuda",

    #Samoa
    "Apia":"Samoa",

    #England
    "Arundel":"England",
    "Beckenham":"England",
    "Blackpool":"England",
    "Brighton":"England",
    "Chelmsford":"England",
    "Cheltenham":"England",
    "Chesterfield":"England",
    "Coggeshall":"England",
    "Colchester":"England",
    "Derby":"England",
    "Bishop's Stortford":"England",
    "Easterbourne":"England",
    "Frinton-on-Sea":"England",
    "Gosforth":"England",
    "Grantham":"England",
    "Guildford":"England",
    "Halstead":"England",
    "Horsham":"England",
    "Hove":"England",

    #Guernsey
    "Castel":"Guernsey",
    

    #Indonesia
    "Bali":"Indonesia",

    #Malaysia
    "Bandar Kinrara":"Malaysia",
    "Bangi":"Malaysia",

    #Thailand
    "Bangkok":"Thailand",
    "Bangi":"Thailand",
    "Chiang Mai":"Thailand",

    #Barbados
    "Barbados":"Barbados",

    #Serbia
    "Belgrade":"Serbia",

    #Bermuda
    "Bermuda":"Bermuda",

    #Malawi
    "Blantyre":"Malawi",

    #Bangladesh
    "Bogra":"Bangladesh",

    #Northern Ireland
    "Bready":"Northern Ireland",
    "Comber":"Northern Ireland",
    "Derry":"Northern Ireland",
    "Eglinton":"Northern Ireland",

    #Uganda
    "Entebbe":"Uganda",
    "Jinja":"Uganda",

    #Cyprus
    "Episkopi":"Cyprus",
    

    #Denmark
    "Brondby":"Denmark",
    "Copenhagen":"Denmark",

    #Argentina
    "Buenos Aires":"Argentina",

    #Austria
    "Graz":"Austria",

    #Grenada
    "Grenada":"Grenada",

    #Saint Lucia
    "Gros Islet":"Saint Lucia",

    #Costa Rica
    "Guacima":"Costa Rica",

    #guyana
    "Guyana":"Guyana",

    #China
    "Hangzhou":"China",

    #Hong Hong
    "Hong Hong":"Hong Hong",

    #Romania
    "Ilfov County":"Romania",

    #South Korea
    "Incheon":"South Korea",

    #Denmark
    "Ishoj":"Denmark",
    

    #India
    "Chandigarh":"India",
    "Dehra Dun":"India",
    "Dharamsala":"India",
    "Faridabad":"India",
    "Gwalior":"India",
    "Jamshedpur":"India",

    #Qatar
    "Doha":"Qatar",

    #Belgium
    "Gelephu":"Belgium",

    #Gibraltar
    "Gibraltar":"Gibraltar",
    
    #Doinica
    "Dominica":"Dominica",

    #France
    "Dreux":"France",

    #South Africa
    "East London":"South Africa",
    

    #Wales
    "Colwyn Bay":"Wales",

    #Tanzania
    "Dar-es-Salaam":"Tanzania",

    #Philippines
    "Dasmarinas":"Philippines"
}
location_df["Country"]=location_df["Country"].fillna(
    location_df["City"].map(country_map2)
)
country_map1.update(country_map2)

    
    

In [207]:
unmatched=location_df[location_df["Country"].isna()]
print("Remaining unmatched:",len(unmatched))

Remaining unmatched: 146


In [209]:
location_df.to_csv(r"C:\Users\R Sujit\Downloads\location_dataset.csv",index=False)
print("Location dataset saved successfully!")

Location dataset saved successfully!


In [211]:
#climate zone mapping 
climate_map={
    "Australia":"Warm Temperate",
    "India":"Tropical",
    "England":"Oceanic",
    "Scotland":"Oceanic",
    "Wales":"Oceanic",
    "Ireland":"Oceanic",
    "Northern Ireland":"Oceanic",
    "New Zealand":"Oceanic",
    "South Africa":"Mediterranean",
    "Pakistan":"Semi-Arid",
    "Sri Lanka":"Tropical",
    "Bangladesh":"Tropical Monsoon",
    "Zimbabwe":"Tropical",
    "United Arab Emirates":"Desert",
    "United States":"Temperate",
    "West Indies":"Tropical",
    "Barbados":"Tropical",
    "Jamaica":"Tropical",
    "Guyana":"Tropical",
    "Antigua and Barbuda":"Tropical",
    "Saint Vincent and the Grenadines":"Tropical",
    "Saint Lucis":"Tropical",
    "Nepal":"Temperate",
    "Netherlands":"Oceanic",
    "Namibia":"Desert",
    "Oman":"Desert",
    "Qatar":"Desert",
    "Kuwait":"Desert",
    "Malaysia":"Tropical",
    "Singapore":"Tropical",
    "Hong Hong":"Subtropical",
    "Thailand":"Tropical",
    "Indonesia":"Tropical",
    "Botswana":"Semi-Arid",
    "Nigeria":"Tropical",
    "Uganda":"Tropical",
    "Rwanda":"Tropical",
    "Kenya":"Tropical",
    "Canada":"Cool Temperate",
    "Unknown":"Temperate",
    "Ghana":"Tropical",
    "Portugal":"Mediterranean",
    "Belgium":"Oceanic",
    "France":"Oceanic",
    "Austria":"Cool Temperate",
    "Germany":"Oceanic",
    "Denmark":"Oceanic",
    "Finland":"Cool Temperate",
    "Norway":"Cool Temperate",
    "Sweden":"Cool Temperate",
    "Netherlands":"Oceanic",
    "Spain":"Mediterranean",
    "Italy":"Meditterranean",
    "Romania":"Temperate",
    "Cyprus":"Mediterranean",
    "Malta":"Mediterranean",
    "Hong kong":"Subtropical",
    "China":"Temperate",
    "South Korea":"Temperate",
    "Japan":"Temperate",
    "Philippines":"Tropical",
    "Cayman Islands":"Tropical",
    "Guernsey":"Oceanic",
    "Bermuda":"Subtropical"
    
}
location_df["Country"]=location_df["Country"].fillna("Unknown")
location_df["Climate_Zone"]=location_df["Country"].map(climate_map)
location_df.head(10)

,City,Country,Climate_Zone
0,Aberdeen,Scotland,Oceanic
1,Abu Dhabi,United Arab Emirates,Desert
2,Abuja,Nigeria,Tropical
3,Accra,Ghana,Tropical
4,Adelaide,Australia,Warm Temperate
5,Ahmedabad,India,Tropical
6,Al Amarat,Oman,Desert
7,Albergaria,Portugal,Mediterranean
8,Albury,Australia,Warm Temperate
9,Alexandra,New Zealand,Oceanic


In [214]:
def generate_weather(climate):
    if climate=="Tropical":
        return(
            np.random.randint(28,39),
            np.random.randint(70,96),
            np.random.randint(5,21),
            np.random.randint(40,91),
            np.random.choice(["Yes","No"],p=[0.6,0.4])
        )
    elif climate=="Tropical Monsoon":
        return(
            np.random.randint(26,36),
            np.random.randint(80,99),
            np.random.randint(5,26),
            np.random.randint(60,101),
            np.random.choice(["Yes","No"],p=[0.8,0.2])
        )
    elif climate=="Desert":
        return(
            np.random.randint(30,46),
            np.random.randint(20,51),
            np.random.randint(10,31),
            np.random.randint(0,31),
            np.random.choice(["Yes","No"],p=[0.1,0.9])
        )
    elif climate=="Oceanic":
        return(
            np.random.randint(8,23),
            np.random.randint(60,96),
            np.random.randint(10,31),
            np.random.randint(50,101),
            np.random.choice(["Yes","No"],p=[0.7,0.3])
        )
    elif climate=="Warm Temperate":
        return(
            np.random.randint(15,31),
            np.random.randint(40,81),
            np.random.randint(5,21),
            np.random.randint(20,71),
            np.random.choice(["Yes","No"],p=[0.4,0.6])
        )
    elif climate=="Cool Temperate":
        return(
            np.random.randint(5,21),
            np.random.randint(50,86),
            np.random.randint(10,26),
            np.random.randint(30,81),
            np.random.choice(["Yes","No"],p=[0.6,0.4])
        )
    elif climate=="Mediterranean":
        return(
            np.random.randint(18,36),
            np.random.randint(30,71),
            np.random.randint(5,21),
            np.random.randint(10,51),
            np.random.choice(["Yes","No"],p=[0.2,0.8])
        )
    elif climate=="Semi-Arid":
        return(
            np.random.randint(22,38),
            np.random.randint(30,61),
            np.random.randint(8,61),
            np.random.randint(10,51),
            np.random.choice(["Yes","No"],p=[0.3,0.7])
        )
    elif climate=="Subtropical":
        return(
            np.random.randint(20,34),
            np.random.randint(60,91),
            np.random.randint(5,21),
            np.random.randint(30,81),
            np.random.choice(["Yes","No"],p=[0.5,0.5])
        )
    else:
        return(
            np.random.randint(15,31),
            np.random.randint(40,81),
            np.random.randint(5,21),
            np.random.randint(20,71),
            "No"
        )
location_df[
    ["Temperature",
     "Humidity",
     "Wind_Speed",
     "Cloud_Cover",
     "Rainfall"]
]=location_df["Climate_Zone"].apply(generate_weather).apply(pd.Series)
location_df.head(10)
            

,City,Country,Climate_Zone,Temperature,Humidity,Wind_Speed,Cloud_Cover,Rainfall
0,Aberdeen,Scotland,Oceanic,17,92,10,74,No
1,Abu Dhabi,United Arab Emirates,Desert,32,22,26,2,No
2,Abuja,Nigeria,Tropical,38,82,18,69,No
3,Accra,Ghana,Tropical,30,90,12,40,No
4,Adelaide,Australia,Warm Temperate,21,52,5,64,No
5,Ahmedabad,India,Tropical,38,83,9,49,No
6,Al Amarat,Oman,Desert,35,38,28,11,No
7,Albergaria,Portugal,Mediterranean,24,47,11,12,No
8,Albury,Australia,Warm Temperate,20,40,14,59,No
9,Alexandra,New Zealand,Oceanic,8,94,12,79,No


In [215]:
location_df.to_csv(r"C:\Users\R Sujit\Downloads\weather_dataset.csv",
                   index=False
                  )
print("Weather Dataset saved successfully!")

Weather Dataset saved successfully!


In [216]:
match_df.columns
match_df.head()

,Match_ID,Date,Num_Days,Match_Type,Gender,Team1,Team2,Venue,City,Season,...,Overs,Toss_Winner,Toss_Decision,Winner,Result_Type,Win_By_Runs,Win_By_Wickets,Player_of_Matche,Num_Innings,Umpires
0,1000851,2016-11-03,5,Test,male,Australia,South Africa,None,Perth,2016/17,...,NaN,South Africa,bat,South Africa,None,177.0,NaN,K Rabada,4,"Aleem Dar,NJ Llong"
1,1000853,2016-11-12,4,Test,male,Australia,South Africa,None,Hobart,2016/17,...,NaN,South Africa,field,South Africa,None,80.0,NaN,KJ Abbott,3,"Aleem Dar,RA Kettleborough"
2,1000855,2016-11-24,4,Test,male,Australia,South Africa,None,None,2016/17,...,NaN,South Africa,bat,Australia,None,NaN,7.0,UT Khawaja,4,"RA Kettleborough,NJ Llong"
3,1000881,2016-12-15,5,Test,male,Australia,Pakistan,None,Brisbane,2016/17,...,NaN,Australia,bat,Australia,None,39.0,NaN,Asad Shafiq,4,"IJ Gould,RK Illingworth"
4,1000883,2016-12-26,5,Test,male,Australia,Pakistan,None,None,2016/17,...,NaN,Pakistan,bat,Australia,None,18.0,NaN,SPD Smith,3,"IJ Gould,S Ravi"


In [218]:
#merging the match dataset and weather 
match_weather_df=pd.merge(
    match_df,
    location_df[
    [
        "City",
        "Country",
        "Climate_Zone",
        "Temperature",
        "Humidity",
        "Wind_Speed",
        "Cloud_Cover",
        "Rainfall"
    ]
    ],
    on="City",
    how="left"
)
match_weather_df.head()

,Match_ID,Date,Num_Days,Match_Type,Gender,Team1,Team2,Venue,City,Season,...,Player_of_Matche,Num_Innings,Umpires,Country,Climate_Zone,Temperature,Humidity,Wind_Speed,Cloud_Cover,Rainfall
0,1000851,2016-11-03,5,Test,male,Australia,South Africa,None,Perth,2016/17,...,K Rabada,4,"Aleem Dar,NJ Llong",Australia,Warm Temperate,19,76,12,23,No
1,1000853,2016-11-12,4,Test,male,Australia,South Africa,None,Hobart,2016/17,...,KJ Abbott,3,"Aleem Dar,RA Kettleborough",Australia,Warm Temperate,29,74,12,42,Yes
2,1000855,2016-11-24,4,Test,male,Australia,South Africa,None,None,2016/17,...,UT Khawaja,4,"RA Kettleborough,NJ Llong",Unknown,Temperate,20,75,13,67,No
3,1000881,2016-12-15,5,Test,male,Australia,Pakistan,None,Brisbane,2016/17,...,Asad Shafiq,4,"IJ Gould,RK Illingworth",Australia,Warm Temperate,30,75,6,50,Yes
4,1000883,2016-12-26,5,Test,male,Australia,Pakistan,None,None,2016/17,...,SPD Smith,3,"IJ Gould,S Ravi",Unknown,Temperate,20,75,13,67,No


In [219]:
print(match_weather_df.isna().sum())

Match_ID                0
Date                    0
Num_Days                0
Match_Type              0
Gender                  0
Team1                   0
Team2                   0
Venue               17824
City                 1490
Season                  0
Event_Name          17824
Balls_per_over          0
Overs                3065
Toss_Winner             0
Toss_Decision           0
Winner               1551
Result_Type         16273
Win_By_Runs          9719
Win_By_Wickets       9661
Player_of_Matche        0
Num_Innings             0
Umpires                 0
Country                 0
Climate_Zone          287
Temperature             0
Humidity                0
Wind_Speed              0
Cloud_Cover             0
Rainfall                0
dtype: int64


In [220]:
match_weather_df[match_weather_df["Climate_Zone"].isna()]["Country"].value_counts()

Country
Saint Kitts and Nevis    66
Argentina                51
Saint Lucia              40
Trinidad and Tobago      31
Samoa                    18
Gibraltar                17
Grenada                  16
Tanzania                 14
Malawi                   14
Dominica                 11
Serbia                    5
Costa Rica                4
Name: count, dtype: int64

In [221]:
climate_map.update({
    "Saint Kitts and Nevis":"Tropical",
    "Argentina":"Temperate",
    "Saint Lucia":"Tropical",
    "Trinidad and Tobago":"Tropical",
    "Samoa":"Tropical",
    "Gibraltar":"Mediterranean",
    "Grenada":"Tropical",
    "Tanzania":"Tropical",
    "Malawi":"Tropical",
    "Dominica":"Tropical",
    "Serbia":"Temperate",
    "Costa Rica":"Tropical"
})

In [224]:
match_weather_df["Climate_Zone"]=match_weather_df["Country"].map(climate_map)
print(match_weather_df["Climate_Zone"].isna().sum())

0


In [225]:
match_weather_df[
    ["Temperature",
     "Humidity",
     "Wind_Speed",
     "Cloud_Cover",
     "Rainfall"]
]=match_weather_df["Climate_Zone"].apply(generate_weather).apply(pd.Series)
match_weather_df.head()

,Match_ID,Date,Num_Days,Match_Type,Gender,Team1,Team2,Venue,City,Season,...,Player_of_Matche,Num_Innings,Umpires,Country,Climate_Zone,Temperature,Humidity,Wind_Speed,Cloud_Cover,Rainfall
0,1000851,2016-11-03,5,Test,male,Australia,South Africa,None,Perth,2016/17,...,K Rabada,4,"Aleem Dar,NJ Llong",Australia,Warm Temperate,18,47,7,48,Yes
1,1000853,2016-11-12,4,Test,male,Australia,South Africa,None,Hobart,2016/17,...,KJ Abbott,3,"Aleem Dar,RA Kettleborough",Australia,Warm Temperate,28,51,18,47,No
2,1000855,2016-11-24,4,Test,male,Australia,South Africa,None,None,2016/17,...,UT Khawaja,4,"RA Kettleborough,NJ Llong",Unknown,Temperate,19,49,17,21,No
3,1000881,2016-12-15,5,Test,male,Australia,Pakistan,None,Brisbane,2016/17,...,Asad Shafiq,4,"IJ Gould,RK Illingworth",Australia,Warm Temperate,15,61,5,35,No
4,1000883,2016-12-26,5,Test,male,Australia,Pakistan,None,None,2016/17,...,SPD Smith,3,"IJ Gould,S Ravi",Unknown,Temperate,27,49,19,58,No


In [226]:
match_weather_df[
    [
        "City",
        "Country",
        "Climate_Zone",
        "Temperature",
        "Humidity",
        "Wind_Speed",
        "Cloud_Cover",
        "Rainfall"
    ]
].head(10)

,City,Country,Climate_Zone,Temperature,Humidity,Wind_Speed,Cloud_Cover,Rainfall
0,Perth,Australia,Warm Temperate,18,47,7,48,Yes
1,Hobart,Australia,Warm Temperate,28,51,18,47,No
2,None,Unknown,Temperate,19,49,17,21,No
3,Brisbane,Australia,Warm Temperate,15,61,5,35,No
4,None,Unknown,Temperate,27,49,19,58,No
5,None,Unknown,Temperate,28,46,17,20,No
6,Brisbane,Australia,Warm Temperate,24,44,7,32,No
7,None,Unknown,Temperate,28,71,15,50,No
8,Perth,Australia,Warm Temperate,19,76,6,57,No
9,None,Unknown,Temperate,17,59,6,61,No


In [227]:
match_weather_df.to_csv(r"C:\Users\R Sujit\OneDrive\Desktop\AI-Driven Cricket Team Selection Project\datasets\weather data\match_weather_dataset.csv",
                        index=False
                       )
print("Match+Weather dataset saved successfully!")

Match+Weather dataset saved successfully!


In [229]:
player_stats.head()
player_stats.columns

Index(['Player', 'Matches', 'Innings', 'Runs', 'Balls', 'Fours', 'Sixes',
       'Outs', 'Not_Outs', 'Batting_Average', 'Strike_Rate',
       'Boundary_Percentage', 'Bowling_Matches', 'Balls_Bowled',
       'Runs_Conceded', 'Wickets', 'Bowling_Average', 'Economy',
       'Fielding_Matches', 'Catches', 'Run_Ounts', 'Stumpings'],
      dtype='object')

In [230]:
np.random.seed(42)
fitness_df=pd.DataFrame()
fitness_df["Player"]=player_stats["Player"]

fitness_df["Fitness_Score"]=np.random.randint(70,101,len(fitness_df))
fitness_df["Injury_Status"]=np.random.choice(
    ["No","Minor","Major"],
    size=len(fitness_df),
    p=[0.80,0.15,0.05]
)
fitness_df["Fatigue_Level"]=np.random.randint(5,41,len(fitness_df))
fitness_df.head()

,Player,Fitness_Score,Injury_Status,Fatigue_Level
0,A Adey,76,No,24
1,A Ahmadhel,89,Minor,17
2,A Alexander,98,No,28
3,A Amado,84,No,11
4,A Amburose,80,Minor,12


In [232]:
fitness_df["Match_Readiness"]=(
    fitness_df["Fitness_Score"]
    -(fitness_df["Fatigue_Level"]*0.5)
)
#reduce readiness based on injury
fitness_df.loc[
   fitness_df["Injury_Status"]=="Minor",
   "Match_Readiness"
]-=15
fitness_df.loc[
   fitness_df["Injury_Status"]=="Major",
   "Match_Readiness"
]-=35

fitness_df["Match_Readiness"]=(
    fitness_df["Match_Readiness"]
    .clip(0,100)
    .round(0)
    .astype(int)
)
fitness_df.head()


,Player,Fitness_Score,Injury_Status,Fatigue_Level,Match_Readiness
0,A Adey,76,No,24,64
1,A Ahmadhel,89,Minor,17,66
2,A Alexander,98,No,28,84
3,A Amado,84,No,11,78
4,A Amburose,80,Minor,12,59


In [234]:
fitness_df.to_csv(
    r"C:\Users\R Sujit\Downloads\fitness_dataset.csv",
    index=False
)
print("Fitness Dataset saved successfully!")

Fitness Dataset saved successfully!


In [235]:
player_final_df=pd.merge(
    player_stats,
    fitness_df,
    on="Player",
    how="left"
)
player_final_df.head()

,Player,Matches,Innings,Runs,Balls,Fours,Sixes,Outs,Not_Outs,Batting_Average,...,Bowling_Average,Economy,Fielding_Matches,Catches,Run_Ounts,Stumpings,Fitness_Score,Injury_Status,Fatigue_Level,Match_Readiness
0,A Adey,1.0,1.0,13.0,17.0,2.0,0.0,1.0,0.0,13.0,...,34.00,11.33,1.0,1.0,0.0,0.0,76,No,24,64
1,A Ahmadhel,4.0,4.0,8.0,27.0,1.0,0.0,4.0,0.0,2.0,...,21.00,10.50,1.0,1.0,0.0,0.0,89,Minor,17,66
2,A Alexander,8.0,8.0,50.0,35.0,2.0,4.0,5.0,3.0,10.0,...,11.67,5.83,10.0,9.0,2.0,0.0,98,No,28,84
3,A Amado,2.0,2.0,19.0,28.0,2.0,0.0,2.0,0.0,9.5,...,20.00,6.67,2.0,2.0,0.0,0.0,84,No,11,78
4,A Amburose,2.0,2.0,0.0,6.0,0.0,0.0,2.0,0.0,0.0,...,0.00,0.00,0.0,0.0,0.0,0.0,80,Minor,12,59


In [236]:
player_final_df.isna().sum()

Player                 0
Matches                0
Innings                0
Runs                   0
Balls                  0
Fours                  0
Sixes                  0
Outs                   0
Not_Outs               0
Batting_Average        0
Strike_Rate            0
Boundary_Percentage    0
Bowling_Matches        0
Balls_Bowled           0
Runs_Conceded          0
Wickets                0
Bowling_Average        0
Economy                0
Fielding_Matches       0
Catches                0
Run_Ounts              0
Stumpings              0
Fitness_Score          0
Injury_Status          0
Fatigue_Level          0
Match_Readiness        0
dtype: int64

In [237]:
player_final_df.to_csv(
    r"C:\Users\R Sujit\Downloads\player_final_dataset.csv",
    index=False
)
print("Player Final Dataset saved successfully!")

Player Final Dataset saved successfully!


In [240]:
match["info"]["players"]

{'Scotland': ['KJ Coetzer',
  'CD Wallace',
  'CS MacLeod',
  'PL Mommsen',
  'RD Berrington',
  'CD de Lange',
  'MH Cross',
  'MA Leask',
  'SM Sharif',
  'AC Evans',
  'CB Sole'],
 'United Arab Emirates': ['Rohan Mustafa',
  'L Sreekumar',
  'Mohammad Shahzad',
  'Shaiman Anwar',
  'Muhammad Usman',
  'Rameez Shahzad',
  'Amjad Javed',
  'Saqlain Haider',
  'Fayyaz Ahmed',
  'Mohammad Naveed',
  'Ahmed Raza']}

In [262]:
match_player_data=[]
for match_index,match in enumerate(all_matches):
    match_id=match_df.iloc[match_index]["Match_ID"]
    date=match["info"]["dates"][0]
    players=match["info"].get("players",{})
    for team,team_players in players.items():
        for player in team_players:
            match_player_data.append({
                "Match_ID":match_id,
                "Date":date,
                "Team":team,
                "Player":player
            })
match_player_df=pd.DataFrame(match_player_data)
match_player_df.head()            

,Match_ID,Date,Team,Player
0,1000851,2016-11-03,Australia,DA Warner
1,1000851,2016-11-03,Australia,SE Marsh
2,1000851,2016-11-03,Australia,UT Khawaja
3,1000851,2016-11-03,Australia,SPD Smith
4,1000851,2016-11-03,Australia,AC Voges


In [263]:
print(player_final_df["Player"].head(20))
print(match_player_df["Player"].head(20))

0             A Adey
1         A Ahmadhel
2        A Alexander
3            A Amado
4         A Amburose
5     A Anandakumara
6          A Andrews
7         A Anlezark
8     A Aravinddaraj
9     A Ashish Reddy
10           A Ashok
11         A Ashokan
12        A Athanaze
13        A Augastin
14          A Austin
15         A Awasthi
16            A Ayre
17          A Badoni
18           A Bagai
19       A Balbirnie
Name: Player, dtype: object
0        DA Warner
1         SE Marsh
2       UT Khawaja
3        SPD Smith
4         AC Voges
5         MR Marsh
6        PM Nevill
7         MA Starc
8        PM Siddle
9     JR Hazlewood
10         NM Lyon
11         SC Cook
12         D Elgar
13         HM Amla
14       JP Duminy
15    F du Plessis
16        T Bavuma
17       Q de Kock
18    VD Philander
19      KA Maharaj
Name: Player, dtype: object


In [264]:
common_players=set(player_final_df["Player"])&set(match_player_df["Player"])
print("Common Players:",len(common_players))
print("Player Final Dataset:",player_final_df["Player"].nunique())
print("Match Player Dataset:",match_player_df["Player"].nunique())
len(common_players)

Common Players: 9789
Player Final Dataset: 9836
Match Player Dataset: 9880


9789

In [265]:
final_ml_df=pd.merge(
    match_player_df,
    player_final_df,
    on="Player",
    how="left"
)
final_ml_df.head()


,Match_ID,Date,Team,Player,Matches,Innings,Runs,Balls,Fours,Sixes,...,Bowling_Average,Economy,Fielding_Matches,Catches,Run_Ounts,Stumpings,Fitness_Score,Injury_Status,Fatigue_Level,Match_Readiness
0,1000851,2016-11-03,Australia,DA Warner,655.0,758.0,28573.0,29705.0,3117.0,628.0,...,71.50,4.88,276.0,349.0,25.0,0.0,77.0,No,14.0,70.0
1,1000851,2016-11-03,Australia,SE Marsh,323.0,374.0,13289.0,17661.0,1400.0,238.0,...,0.00,0.00,102.0,132.0,7.0,0.0,73.0,No,27.0,60.0
2,1000851,2016-11-03,Australia,UT Khawaja,300.0,404.0,14426.0,23792.0,1626.0,113.0,...,0.00,3.25,123.0,163.0,6.0,0.0,86.0,No,19.0,76.0
3,1000851,2016-11-03,Australia,SPD Smith,519.0,627.0,23632.0,32966.0,2339.0,344.0,...,34.11,5.14,287.0,478.0,33.0,0.0,75.0,No,26.0,62.0
4,1000851,2016-11-03,Australia,AC Voges,147.0,170.0,5381.0,7657.0,541.0,32.0,...,37.62,5.34,62.0,83.0,5.0,0.0,70.0,No,28.0,56.0


In [266]:
match_weather_df.columns.tolist()

['Match_ID',
 'Date',
 'Num_Days',
 'Match_Type',
 'Gender',
 'Team1',
 'Team2',
 'Venue',
 'City',
 'Season',
 'Event_Name',
 'Balls_per_over',
 'Overs',
 'Toss_Winner',
 'Toss_Decision',
 'Winner',
 'Result_Type',
 'Win_By_Runs',
 'Win_By_Wickets',
 'Player_of_Matche',
 'Num_Innings',
 'Umpires',
 'Country',
 'Climate_Zone',
 'Temperature',
 'Humidity',
 'Wind_Speed',
 'Cloud_Cover',
 'Rainfall']

In [267]:
print(final_ml_df["Match_ID"].dtype)
print(match_weather_df["Match_ID"].dtype)

object
int64


In [268]:
final_ml_df["Match_ID"]=final_ml_df["Match_ID"].astype(int)
match_weather_df["Match_ID"]=match_weather_df["Match_ID"].astype(int)

In [269]:
weather_columns=[
    "Match_ID",
    "Match_Type",
    "City",
    "Country",
    "Climate_Zone",
    "Temperature",
    "Humidity",
    "Wind_Speed",
    "Cloud_Cover",
    "Rainfall"
]
final_ml_df=pd.merge(
    final_ml_df,
    match_weather_df[weather_columns],
    on="Match_ID",
    how="left"
)

In [270]:
print(final_ml_df.shape)
final_ml_df.head()
final_ml_df.isna().sum()

(393419, 38)


Match_ID                   0
Date                       0
Team                       0
Player                     0
Matches                  117
Innings                  117
Runs                     117
Balls                    117
Fours                    117
Sixes                    117
Outs                     117
Not_Outs                 117
Batting_Average          117
Strike_Rate              117
Boundary_Percentage      117
Bowling_Matches          117
Balls_Bowled             117
Runs_Conceded            117
Wickets                  117
Bowling_Average          117
Economy                  117
Fielding_Matches         117
Catches                  117
Run_Ounts                117
Stumpings                117
Fitness_Score            117
Injury_Status            117
Fatigue_Level            117
Match_Readiness          117
Match_Type                 0
City                   32825
Country                    0
Climate_Zone               0
Temperature                0
Humidity      

In [271]:
missing_players=final_ml_df[final_ml_df["Matches"].isna()]
print("Missing Players:",missing_players["Player"].nunique())
missing_players[["Player","Team"]].drop_duplicates().head(30)

Missing Players: 91


,Player,Team
5013,Farhat Mahmood,Spain
6341,M Morettini,Italy
16179,P Pritchard,Samoa
35027,SA Gan,Tripura
40047,Abbas Saad,Saudi Arabia
40773,N Cherry,Gambia
47517,J Hoffman,Czech Republic
49739,L Chetri,Meghalaya
49922,PS Chaitanya Reddy,Hyderabad (India)
49935,Bhiguraj Pathania,Uttarakhand


In [272]:
final_ml_df=final_ml_df.dropna(
    subset=[
        "Matches",
        "Fitness_Score",
        "Match_Readiness"
    ]
)
print(final_ml_df.shape)

(393302, 38)


In [273]:
final_ml_df["City"]=final_ml_df["City"].fillna("Unknown")
final_ml_df.isna().sum()

Match_ID               0
Date                   0
Team                   0
Player                 0
Matches                0
Innings                0
Runs                   0
Balls                  0
Fours                  0
Sixes                  0
Outs                   0
Not_Outs               0
Batting_Average        0
Strike_Rate            0
Boundary_Percentage    0
Bowling_Matches        0
Balls_Bowled           0
Runs_Conceded          0
Wickets                0
Bowling_Average        0
Economy                0
Fielding_Matches       0
Catches                0
Run_Ounts              0
Stumpings              0
Fitness_Score          0
Injury_Status          0
Fatigue_Level          0
Match_Readiness        0
Match_Type             0
City                   0
Country                0
Climate_Zone           0
Temperature            0
Humidity               0
Wind_Speed             0
Cloud_Cover            0
Rainfall               0
dtype: int64

In [274]:
final_ml_df.to_csv(
    r"C:\Users\R Sujit\Downloads\final_ml_dataset.csv",
    index=False
)
print("Final ML Dataset cleaned and saved successfully!")

Final ML Dataset cleaned and saved successfully!
